# Create SQL Views in DBRepo

Creates the three views defined in [`../data/views.sql`](../data/views.sql) in the SDCC Traffic Flow Experiment 2022 database on DBRepo.

All three views are denormalising joins of `TrafficMeasurements` with `Calendar` and `TrafficSites`, two of them with simple WHERE filters. They are expressible in DBRepo's structured `QueryDefinition` model — feature engineering (cyclical time, target binning, class balancing, hourly aggregation) is done downstream in Python.

Prerequisite: the schema and data must already be loaded — see [`upload-data-DBrepo-notebook.ipynb`](upload-data-DBrepo-notebook.ipynb).

DBRepo API documentation: https://www.ifs.tuwien.ac.at/infrastructures/dbrepo/dbrepo_v1.13.3.pdf

> **⚠ Blocked on upstream fixes.** This notebook depends on two changes to [`upload-data-DBrepo-notebook.ipynb`](upload-data-DBrepo-notebook.ipynb): (1) include `start_time` in the `TrafficMeasurements` upload, (2) make NaN-date handling consistent between `Calendar` and `TrafficMeasurements`. See [`../docs/pending-fixes.md`](../docs/pending-fixes.md) for the full checklist. Until the re-upload happens, `v_measurements_enriched` and `v_peak_hour_measurements` will fail with a missing-column error on `start_time`.

## 1. Set up the REST client

In [ ]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    JoinDefinition,
    JoinType,
    ConditionalDefinition,
    FilterDefinition,
    FilterType,
)
from getpass import getpass

In [ ]:
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID     = "2de50b61-ac24-4484-8117-dc0fe9dc1b7c"

username = "e11705340"
user_password = getpass(f"Enter password for {username}: ")

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=username,
    password=user_password,
)

## 2. Discover the tables' internal names

DBRepo generates a normalised `internal_name` for each table when it is created (typically lowercase, with non-ASCII replaced). `QueryDefinition` references tables and columns by these internal names in `"table.column"` form, so we look them up here once and reuse below.

In [ ]:
database = client.get_database(database_id=DATABASE_ID)
tables_by_display_name = {t.name: t.internal_name for t in database.tables}
print(tables_by_display_name)

T_MEAS = tables_by_display_name["TrafficMeasurements"]
T_CAL  = tables_by_display_name["Calendar"]
T_SITE = tables_by_display_name["TrafficSites"]

## 3. Create the views

Each view is built from a `QueryDefinition`. The Python SDK converts the structured definition into the database-level SQL view DBRepo registers. If you need to re-run after editing a view, use section 4 to drop it first.

In [ ]:
# Common join shape reused by all three views.
join_calendar = JoinDefinition(
    type=JoinType.INNER,
    datasource=T_CAL,
    conditionals=[
        ConditionalDefinition(
            column=f"{T_MEAS}.date_id",
            foreign_column=f"{T_CAL}.date_id",
        )
    ],
)

join_sites = JoinDefinition(
    type=JoinType.INNER,
    datasource=T_SITE,
    conditionals=[
        ConditionalDefinition(
            column=f"{T_MEAS}.site_id",
            foreign_column=f"{T_SITE}.site_id",
        )
    ],
)

# Columns the views expose, in display order.
view_columns = [
    f"{T_MEAS}.observation_id",
    f"{T_MEAS}.site_id",
    f"{T_MEAS}.date_id",
    f"{T_CAL}.day_of_week",
    f"{T_MEAS}.start_time",
    f"{T_MEAS}.end_time",
    f"{T_MEAS}.flow",
    f"{T_MEAS}.flow_pc",
    f"{T_MEAS}.cong",
    f"{T_MEAS}.cong_pc",
    f"{T_MEAS}.dsat",
    f"{T_MEAS}.dsat_pc",
]

In [ ]:
# 1. v_measurements_enriched -- full denormalised join, no filter.
client.create_view(
    database_id=DATABASE_ID,
    name="v_measurements_enriched",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
    ),
    is_public=True,
    is_schema_public=True,
)

In [ ]:
# 2. v_weekday_measurements -- same join, filter day_of_week to Mon..Fri.
client.create_view(
    database_id=DATABASE_ID,
    name="v_weekday_measurements",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
        filters=[
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_CAL}.day_of_week",
                operator="!=",
                value="Saturday",
            ),
            FilterDefinition(type=FilterType.AND),
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_CAL}.day_of_week",
                operator="!=",
                value="Sunday",
            ),
        ],
    ),
    is_public=True,
    is_schema_public=True,
)

In [ ]:
# 3. v_peak_hour_measurements -- same join, filter start_time to rush hours.
# Builds: (start_time BETWEEN 07:00 AND 09:00) OR (start_time BETWEEN 17:00 AND 19:00)
client.create_view(
    database_id=DATABASE_ID,
    name="v_peak_hour_measurements",
    query=QueryDefinition(
        columns=view_columns,
        datasources=[T_MEAS],
        joins=[join_calendar, join_sites],
        filters=[
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_MEAS}.start_time",
                operator=">=",
                value="07:00:00",
            ),
            FilterDefinition(type=FilterType.AND),
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_MEAS}.start_time",
                operator="<=",
                value="09:00:00",
            ),
            FilterDefinition(type=FilterType.OR),
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_MEAS}.start_time",
                operator=">=",
                value="17:00:00",
            ),
            FilterDefinition(type=FilterType.AND),
            FilterDefinition(
                type=FilterType.WHERE,
                column=f"{T_MEAS}.start_time",
                operator="<=",
                value="19:00:00",
            ),
        ],
    ),
    is_public=True,
    is_schema_public=True,
)

## 4. Verify

List the views the database now has, then pull a small sample from each.

In [ ]:
views = client.get_views(database_id=DATABASE_ID)
views_by_name = {v.name: v.id for v in views}
for name, vid in views_by_name.items():
    print(f"{name}: {vid}")

In [ ]:
for name in [
    "v_measurements_enriched",
    "v_weekday_measurements",
    "v_peak_hour_measurements",
]:
    if name not in views_by_name:
        print(f"{name}: NOT FOUND")
        continue
    n = client.get_view_data_count(database_id=DATABASE_ID, view_id=views_by_name[name])
    df = client.get_view_data(database_id=DATABASE_ID, view_id=views_by_name[name], size=5)
    print(f"\n--- {name} ({n} rows) ---")
    print(df.head())

## 5. Optional: drop views

Uncomment when you need to recreate a view (e.g. after editing it).

In [ ]:
# for name in [
#     "v_peak_hour_measurements",
#     "v_weekday_measurements",
#     "v_measurements_enriched",
# ]:
#     if name in views_by_name:
#         client.delete_view(database_id=DATABASE_ID, view_id=views_by_name[name])
#         print(f"dropped {name}")